## What We’re Building

We want a chatbot that can answer questions like:

- What does Rambam say about repentance?
- What are the laws of charity?
- How does Rambam describe free will?

Instead of relying only on model memory, we retrieve real Rambam sources first.


## Big Picture: What Is RAG?

RAG stands for **Retrieval-Augmented Generation**.

Instead of asking the LLM to answer from memory:

1. Retrieve relevant sources based on semantic similarity (embedding similarity with query) or exact match filtering
2. Give those sources to the LLM
3. Generate an answer grounded in those sources

The key idea is:

> Retrieve first, generate second.


## RAG Flow for Rambam Q&A

```mermaid
flowchart TD
    A["User Question<br/>What does Rambam say about repentance?"]

    A --> B["Embed Query<br/>q = [0.82, -0.14, 0.33, 0.71]"]

    B --> C["Vector Search<br/>Find K Most Similar Chunks"]

    D1["Rambam Chunk 1<br/>Hilchot Teshuva 2:1<br/>v = [0.80, -0.10, 0.31, 0.69]"]
    D2["Rambam Chunk 2<br/>Hilchot Deot 1:3<br/>v = [0.11, 0.77, -0.22, 0.05]"]
    D3["Rambam Chunk 3<br/>Hilchot Teshuva 7:4<br/>v = [0.79, -0.18, 0.35, 0.73]"]

    D1 --> C
    D2 --> C
    D3 --> C

    C --> E["Similarity Scores<br/>Chunk 1 = 0.998<br/>Chunk 2 = 0.214<br/>Chunk 3 = 0.996"]

    E --> F["Return Top K Matches<br/>K = 2<br/>1. Teshuva 2:1<br/>2. Teshuva 7:4"]

    F --> G["Prompt to LLM<br/>Question + Top K Sources"]

    G --> H["Final Answer<br/>with Rambam citations"]
```


## Note on Re-rankers

The vector search step usually retrieves the **top K most similar chunks** based on embedding similarity.

A re-ranker is an optional extra step after retrieval.

Instead of sending all retrieved chunks directly to the LLM, we can do:

1. Retrieve a larger set of candidates, for example top 10 or top 20  
2. Use a re-ranker to score which chunks are truly most relevant  
3. Send only the best few chunks to the LLM

So the basic flow is:

```text
User Question → Vector Search → Top K Chunks → LLM Answer
```

With a re-ranker:

```text
User Question → Vector Search → Candidate Chunks → Re-ranker → Best Chunks → LLM Answer
```

This can improve answer quality when many chunks are semantically similar but only a few directly answer the question.

## Simpler System Architecture

```mermaid
flowchart LR
    A["Rambam DataFrame"] --> B["Create LlamaIndex Documents"]
    B --> C["Embeddings"]
    C --> D["Vector Index"]

    E["User Question"] --> F["Embed Query"]
    F --> G["Retriever"]
    D --> G

    G --> H["Relevant Rambam Sources"]
    H --> I["LLM"]
    I --> J["Answer with Sources"]
```


## How This Connects to LlamaIndex

Think of LlamaIndex as the **data and retrieval layer** between your Rambam texts and the LLM.

It helps us:

- load text into `Document` objects
- attach metadata
- chunk text into nodes
- create embeddings
- retrieve relevant sources
- optionally re-rank results
- send the best context to the LLM

The LLM still generates the answer, but LlamaIndex helps it use our Rambam data intelligently.


## Documents and Nodes

A `Document` is usually the larger input object we load into LlamaIndex.

LlamaIndex can then split each document into smaller searchable pieces called **Nodes**.

Think of it this way:

- `Document` = original text object we load
- `Node` = smaller chunk used for embeddings and retrieval
- Both can contain metadata such as `ref`, `book`, `hilchot`, `chapter`, and `halacha`

For Rambam, one halacha per document/node is often a strong strategy because retrieved results map cleanly to precise sources.


## Setup


In [ ]:
!pip install -U pandas gradio openai llama-index llama-index-llms-openai llama-index-embeddings-openai


In [ ]:
import os
import pandas as pd
import gradio as gr

from openai import OpenAI

from llama_index.core import (
    VectorStoreIndex,
    Document,
    Settings,
    StorageContext,
    load_index_from_storage
)
from llama_index.core.schema import MetadataMode
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding


## Configure OpenAI + LlamaIndex

We use OpenAI in two places:

1. **Embeddings** — turning Rambam text and user questions into vectors  
2. **LLM responses** — generating the final answer from retrieved sources


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY", "").strip()

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set.")

client = OpenAI(api_key=api_key)

Settings.llm = LlamaOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=api_key
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large",
    api_key=api_key
)


## Convert DataFrame Rows into LlamaIndex Documents

Each row in the dataframe becomes a `Document`.

The `text` field becomes the content.

The other fields become metadata, which lets us cite sources later.


# CODE MYSELF

In [ ]:
def df_to_documents(df):
    documents = []

    for row in df.itertuples(index=False):
        metadata = {
            "ref": row.ref,
            "book": row.book,
            "hilchot": row.hilchot,
            "chapter": int(row.chapter),
            "halacha": int(row.halacha),
            "language": row.language,
            "versionTitle": row.versionTitle,
        }

        documents.append(
            Document(
                text=row.text,
                metadata=metadata
            )
        )

    return documents


In [ ]:
documents = df_to_documents(df)

len(documents), documents[0]


## Build the Vector Index

When we call `VectorStoreIndex.from_documents(...)`, LlamaIndex creates embeddings and stores them in a searchable index.

This is where our Rambam dataframe becomes semantic search.


# CODE MYSELF

In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)


## Save and Load the Index

Persisting the index prevents us from rebuilding embeddings every time.


Load later

## 🧠 Query Engine vs Retriever (RAG Core Components)

### 🔹 Query Engine — “Ask → Answer”

The **query engine** is the high-level interface that produces the final answer.

It handles:

1. Taking retrieved Rambam sources
2. Building a prompt
3. Sending it to the LLM
4. Returning the response

```python
query_engine = index.as_query_engine()

response = query_engine.query(
    "What does Rambam say about repentance?"
)
```

👉 Think:

> **Retriever + Prompt + LLM → Final Answer**

---

### ⚙️ Customizing the Query Engine

```python
query_engine = index.as_query_engine(
    similarity_top_k=6,
    text_qa_template=qa_prompt,
    response_mode="compact",
    node_postprocessors=[reranker],
)
```

You control:

* `text_qa_template` → how the answer is written (grounded, cited, etc.)
* `response_mode` → short vs structured answers
* `similarity_top_k` → how many sources are passed in
* `node_postprocessors` → reranking / refinement

👉 This controls **how the answer is generated**

---

## 🔍 Retriever — “Find the Right Sources”

The **retriever** is the search component.

It does:

1. Embed the query
2. Compare it to stored embeddings
3. Return the top K most relevant chunks

```python
retriever = index.as_retriever(similarity_top_k=5)

nodes = retriever.retrieve(
    "What does Rambam say about free will?"
)
```

👉 Think:

> **A semantic search engine over Rambam**

---

### ⚙️ Customizing the Retriever

```python
from llama_index.core import MetadataFilters, ExactMatchFilter

retriever = index.as_retriever(
    similarity_top_k=5,
    filters=MetadataFilters(filters=[
        ExactMatchFilter(key="hilchot", value="Repentance")
    ])
)
```

You control:

* how many results (`top_k`)
* which parts of Rambam are searched
* filtering by metadata (e.g. Sefer HaMadda only)

👉 This controls **what information enters the system**

---

## 🎯 Key Insight

> The retriever does **not generate the answer**
> but it **determines what goes into the prompt**

So:

> **Better retrieval → better sources → better answer**

---

## 💡 One-Liner (say this in your video)

> “The model can only answer based on what the retriever gives it.”

---

## 🧠 Final Mental Model

| Component    | Role                               |
| ------------ | ---------------------------------- |
| Retriever    | Finds relevant Rambam sources      |
| Query Engine | Turns those sources into an answer |

---

Retriever → raw chunks

Query Engine → chunks + LLM answer

# Query Engine for querying.
* Ask question and get response based on RAG

# CODE MYSELF

Define Query Engine

In [1]:
query_engine=index.as_query_engine()

NameError: name 'index' is not defined

Query it

In [ ]:
query_engine.query("What type of work is permitted on Chol Hamoed?").response

# Retrievers to get Most Similar Nodes + More Granular Control

# CODE MYSELF

## Retrieve Sources Manually

Before using a full query engine, it is helpful to inspect what the retriever returns.

This makes the RAG process less magical.


In [ ]:
def retrieve_sources(index, query, k=5):
    retriever = index.as_retriever(similarity_top_k=k)
    nodes = retriever.retrieve(query)

    rows = []

    for node in nodes:
        md = node.metadata or {}

        rows.append({
            "ref": md.get("ref"),
            "hilchot": md.get("hilchot"),
            "chapter": md.get("chapter"),
            "halacha": md.get("halacha"),
            "score": getattr(node, "score", None),
            "text": node.get_content(metadata_mode=MetadataMode.NONE),
        })

    return pd.DataFrame(rows), nodes


In [ ]:
results_df, nodes = retrieve_sources(
    index,
    "What does Rambam say about free will?",
    k=5
)

results_df


## Optional: Re-rankers

Sometimes the first retrieval pass gives several decent results.

A **re-ranker** can take those candidates and reorder them using a stronger model.

Big picture:

1. Retriever finds top 10 candidates quickly  
2. Re-ranker sorts them more intelligently  
3. Best few sources go to the LLM

For a first version, simple vector retrieval is enough. Later, re-rankers can improve answer quality.


## Grounded Answer Function

Now we manually build the RAG pipeline:

1. Retrieve relevant Rambam sources
2. Insert sources into a prompt
3. Ask the LLM to answer using only those sources
4. Return both the answer and source table


In [ ]:
def ask_rambam_with_sources(index, question, k=5):
    results_df, nodes = retrieve_sources(index, question, k=k)

    if results_df.empty:
        return {
            "answer": "I could not retrieve any relevant Rambam source.",
            "sources": results_df,
        }

    sources = "\n\n".join(
        f"{row['ref']}\n{row['text']}"
        for _, row in results_df.iterrows()
    )

    prompt = f'''
You are a Rambam assistant.

Answer in clear English.
Use ONLY the sources below.
Cite the references clearly.
Do not invent sources.
If the sources only partially answer the question, say so.

Question:
{question}

Sources:
{sources}
'''

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": results_df,
    }


In [ ]:
test = ask_rambam_with_sources(
    index,
    "What does Rambam say about free will?",
    k=5
)

print(test["answer"])
test["sources"]


## Query Engine Shortcut

The manual version is great for teaching because we can inspect the retrieved sources.

But LlamaIndex also gives us a higher-level shortcut:

```python
query_engine = index.as_query_engine()
response = query_engine.query("What does Rambam say about repentance?")
```

The query engine usually does retrieval internally, then sends the retrieved context to the LLM.

Simple mental model:

- Retriever gets the sources
- Query engine turns sources into an answer


In [ ]:
query_engine = index.as_query_engine(similarity_top_k=5)

response = query_engine.query(
    "What does Rambam say about repentance?"
)

print(response)


## Gradio Chat UI

Now we wrap the Q&A function in a simple chat interface.


In [ ]:
def chat_interface(message, history):
    result = ask_rambam_with_sources(
        index,
        message,
        k=5
    )

    return result["answer"]


In [ ]:
demo = gr.ChatInterface(
    fn=chat_interface,
    title="Ask the Rambam",
    description="Ask questions about the Rambam's Mishneh Torah using retrieved sources."
)

demo.launch()


## Final Summary

In this notebook, we started from an existing Rambam dataframe and built a RAG chatbot.

The flow was:

1. DataFrame rows → LlamaIndex Documents  
2. Documents → embeddings  
3. Embeddings → vector index  
4. User question → retrieval  
5. Retrieved sources → LLM answer  
6. Gradio → chat interface

This same pattern can be reused for:

- finance documents
- legal documents
- company knowledge bases
- academic papers
- historical archives
- Torah texts
